In [1]:
import sys; sys.path.insert(0, "/kaggle/working")
import numpy as np
from feature_node import load_data
from feature_edge import add_subsplit, WINDOWS
from paths import TXN_NODES
import node_seq as ns

df     = add_subsplit(load_data())          # ghi đè train_b_mask.npy bằng đúng nội dung cũ
txn    = np.load(TXN_NODES)
n_node = int(txn.max()) + 1

print(f"\nn_node={n_node:,} | BUCKET_H={ns.BUCKET_H}h | K={ns.K} ({ns.K*ns.BUCKET_H}h)")
act_seq = {}
for name, (src_splits, _) in WINDOWS.items():
    Z = ns.build_seq(df, src_splits, txn, n_node)
    act_seq[name] = (Z.reshape(n_node, -1) != 0).any(1).mean() * 100
    print(f"{name:5s}: node hoạt động {act_seq[name]:5.2f}%")
    del Z

  cắt train tại 2022-09-02 12:45:00 | train_a 1,522,941 dòng (pos 524) | train_b 1,523,920 dòng (pos 1,773)
  đã lưu dataset_high\train_b_mask.npy

n_node=515,088 | BUCKET_H=6h | K=4 (24h)
train: node hoạt động 59.15%
val  : node hoạt động 36.30%
test : node hoạt động 38.56%


In [2]:
bucket = df["time"].to_numpy() // (ns.BUCKET_H * 3600 * ns.NS)
ori    = df["ori_idx"].to_numpy()
win    = df["win"].to_numpy()

def active_pct(src_splits, K):
    m = np.isin(win, src_splits)
    b = bucket[m]
    m[m] = b >= b.max() - K + 1
    o = ori[m]
    return len(np.unique(np.concatenate([txn[o, 0], txn[o, 1]]))) / n_node * 100

for name, (src_splits, _) in WINDOWS.items():
    a = active_pct(src_splits, ns.K)
    print(f"{name:5s}: {act_seq[name]:5.2f}% (build_seq) | {a:5.2f}% (đếm trực tiếp)")
    assert abs(act_seq[name] - a) < 1e-9, f"{name}: hai cách đếm lệch nhau"
print("PASS: hai cách đếm độc lập cho cùng một số")

train: 59.15% (build_seq) | 59.15% (đếm trực tiếp)
val  : 36.30% (build_seq) | 36.30% (đếm trực tiếp)
test : 38.56% (build_seq) | 38.56% (đếm trực tiếp)
PASS: hai cách đếm độc lập cho cùng một số


In [3]:
# Cell C — dùng active_pct của Cell B
K_LIST = [4, 6, 8, 12]
print(f"% node hoạt động trong cửa sổ node_seq, theo độ dài cửa sổ (bucket {ns.BUCKET_H}h)")
for K in K_LIST:
    a = {n: active_pct(w[0], K) for n, w in WINDOWS.items()}
    print(f"K={K:2d} ({K*ns.BUCKET_H:3d}h): train {a['train']:5.2f}%  | val {a['val']:5.2f}% | "
          f"test {a['test']:5.2f}%   <-- tỉ lệ {a['train']/a['test']:.2f}x"
          f"{', hiện tại' if K == ns.K else ''}")

% node hoạt động trong cửa sổ node_seq, theo độ dài cửa sổ (bucket 6h)
K= 4 ( 24h): train 59.15%  | val 36.30% | test 38.56%   <-- tỉ lệ 1.53x, hiện tại
K= 6 ( 36h): train 73.81%  | val 42.16% | test 43.01%   <-- tỉ lệ 1.72x
K= 8 ( 48h): train 92.49%  | val 45.37% | test 45.58%   <-- tỉ lệ 2.03x
K=12 ( 72h): train 92.49%  | val 47.27% | test 47.40%   <-- tỉ lệ 1.95x


In [6]:
import numpy as np, pandas as pd, lightgbm as lgb, pyarrow.parquet as pq
from sklearn.metrics import average_precision_score
from paths import OUT_DIR, TX_FEAT, TXN_MATRIX, TXN_NODES

# chép từ check_emb_leak.py — đổi bên đó thì đổi cả bên này
LABEL, HOLD, MAX_RATIO = "Is Laundering", 0.1, 3.0
LGB = {"num_leaves": 63, "learning_rate": 0.05, "n_estimators": 300,
       "max_bin": 63, "n_jobs": -1, "verbose": -1}
D = 32                                   # số chiều embedding của pretrain_gnn

splits = pd.read_parquet(TX_FEAT, columns=["split"])["split"].to_numpy()
nodes  = np.load(TXN_NODES)
mask_b = np.load(f"{OUT_DIR}/train_b_mask.npy")
n_node = int(nodes.max()) + 1

def rows(split):
    m   = splits == split
    idx = np.flatnonzero(m)
    y   = pq.ParquetFile(TXN_MATRIX.format(split)).read(
              columns=[LABEL]).column(0).to_numpy(zero_copy_only=False)
    assert len(idx) == len(y), f"{split}: txn_matrix lệch txn_nodes"
    if split == "train":
        keep = mask_b[m]
        idx, y = idx[keep], y[keep]
    return idx, y.astype("int8")

idx_tr, y_tr = rows("train")
idx_te, y_te = rows("test")
cut = int(len(y_tr) * (1 - HOLD))
print(f"train {cut:,} | holdout {len(y_tr)-cut:,} | test {len(y_te):,} | "
      f"pos holdout={y_tr[cut:].mean():.5f} test={y_te.mean():.5f}")

def ratio_of(Xtr, Xte, name):
    m = lgb.LGBMClassifier(**LGB, random_state=0)
    m.fit(Xtr[:cut], y_tr[:cut])
    L = []
    for X, y in [(Xtr[cut:], y_tr[cut:]), (Xte, y_te)]:
        ap = average_precision_score(y, m.predict_proba(X)[:, 1])
        L.append(ap / y.mean())
    r = L[0] / max(L[1], 1e-9)
    print(f"  {name:32s} độ khuếch đại holdout={L[0]:6.1f}x test={L[1]:5.1f}x -> tỉ lệ {r:5.2f}x")
    return r

def pair(h, idx):
    return np.hstack([h[nodes[idx, 0]], h[nodes[idx, 1]]]).astype("float32")

train 1,371,528 | holdout 152,392 | test 1,015,882 | pos holdout=0.00135 test=0.00177


In [ ]:
# [a] chỉ đo null, 8 seed. [b] và [c] lấy từ lần chạy trước — cùng dữ liệu, không cần chạy lại.
R_CLEAN, R_LEAK = 1.69, 30.33

null = []
for s in range(8):
    h = np.random.default_rng(s).normal(size=(n_node, D)).astype("float32")
    null.append(ratio_of(pair(h, idx_tr), pair(h, idx_te), f"random seed{s}"))
    del h

floor = max(max(null), R_CLEAN)
print(f"\n=== chốt ngưỡng ===")
print(f"  null {len(null)} seed        : {min(null):.2f}x – {max(null):.2f}x "
      f"(trung vị {np.median(null):.2f}x)")
print(f"  đối chứng sạch     : {R_CLEAN:.2f}x")
print(f"  đối chứng dương    : {R_LEAK:.2f}x")
if floor < R_LEAK:
    print(f"\n  -> MAX_RATIO = {np.sqrt(floor * R_LEAK):.1f}    "
          f"(khoảng hợp lệ {floor:.2f}x .. {R_LEAK:.2f}x)")
else:
    print(f"\n  -> KHÔNG có ngưỡng hợp lệ: sàn {floor:.2f}x đã chạm {R_LEAK:.2f}x")

embedding ngẫu nhiên, dùng chung mọi split (không nhãn, không đồ thị)
  random seed0                     độ khuếch đại holdout=   9.8x test=  2.2x -> tỉ lệ  4.50x
  random seed1                     độ khuếch đại holdout=  20.5x test=  2.0x -> tỉ lệ 10.18x
  random seed2                     độ khuếch đại holdout=  22.0x test=  2.0x -> tỉ lệ 11.18x
  random seed3                     độ khuếch đại holdout=  19.3x test=  2.0x -> tỉ lệ  9.72x
  random seed4                     độ khuếch đại holdout=  17.1x test=  2.0x -> tỉ lệ  8.69x
  random seed5                     độ khuếch đại holdout=  34.0x test=  2.5x -> tỉ lệ 13.54x
  random seed6                     độ khuếch đại holdout=  14.5x test=  2.6x -> tỉ lệ  5.62x
  random seed7                     độ khuếch đại holdout=  13.8x test=  1.9x -> tỉ lệ  7.15x

=== chốt ngưỡng ===
  null 8 seed        : 4.50x – 13.54x (trung vị 9.21x)
  đối chứng sạch     : 1.69x
  đối chứng dương    : 30.33x

  -> MAX_RATIO = 20.3    (khoảng hợp lệ 13.54x .. 

In [ ]:
#cell chứng minh tham số tự giám sát không cần tune
from pretrain_gnn import load_graphs, split_edges, MP_FRAC, SUP_FRAC, EDGE_BATCH, STEPS, EVAL_EVERY, PATIENCE, LR

graphs, num_banks, tx_dim = load_graphs()
g_mp, sup, ev = split_edges(graphs["train"], seed=0)

giamsat, n_ev = sup.size(1), ev.size(1)
ratio = EDGE_BATCH / giamsat                 # >=1 nghĩa là mỗi step gần như quét hết không gian giám sát
print(f"tỷ lệ cạnh học mỗi lần chạy trong tổng số cạnh giám sát: {ratio}")
steps_per_pass = giamsat / EDGE_BATCH

print(f"\nLR={LR} | MP_FRAC = {MP_FRAC} | SUP_FRAC = {SUP_FRAC} | EDGE_BATCH={EDGE_BATCH:,} | "
      f"STEPS={STEPS} | EVAL_EVERY={EVAL_EVERY} | PATIENCE={PATIENCE}")

print(f"số cạnh dừng sớm: {n_ev:,}")
if n_ev < 2000:                            # ngưỡng tham khảo, không phải số cứng
    print("  -> khá nhỏ (~<2000), link_ap mỗi lần eval dễ nhiễu => PATIENCE/EVAL_EVERY "
          "có thể dừng sai lúc, cân nhắc tăng phần dư cho ev (giảm MP_FRAC hoặc SUP_FRAC)")
else:
    print("  -> đủ lớn để link_ap ổn định qua các lần eval, MP_FRAC/SUP_FRAC hiện tại ổn")

c:\Users\LENOVO\anaconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  cạnh gốc 3,046,861 (935,111 cặp vô hướng) -> truyền tin 2,591,454 | giám sát 303,089 | dừng sớm 152,318

LR=0.003 | MP_FRAC/SUP_FRAC = 0.85/0.1 | EDGE_BATCH=65,536 | STEPS=600 | EVAL_EVERY=20 | PATIENCE=20
sup (giám sát) = 303,089 cạnh | EDGE_BATCH/sup = 0.216x/step
  -> cần ~4.6 step để quét hết 1 lượt sup => đúng nghĩa minibatch, EDGE_BATCH hợp lý, không cần đụng
ev (dừng sớm) = 152,318 cạnh
  -> đủ lớn để link_ap ổn định qua các lần eval, MP_FRAC/SUP_FRAC hiện tại ổn
patience_window = EVAL_EVERY*PATIENCE = 400 step (~86.5 lượt quét sup) trong tổng 600 step
